In [37]:
requirements = """
gradio
json5
jsonschema
python-dotenv
pydantic
"""

with open("requirements.txt","w") as f:
    f.write(requirements)

print("requirements.txt created successfully")

requirements.txt created successfully


In [38]:
!pip install -q gradio json5 jsonschema python-dotenv pydantic transformers

In [39]:
import gradio
import json5
import jsonschema
import dotenv
import pydantic
import transformers

print("All packages installed successfully ✅")

All packages installed successfully ✅


In [40]:
import json
import re
import logging
from enum import Enum
from typing import Any, Dict

In [41]:
class CaseStyle(Enum):
    SNAKE = "snake_case"
    CAMEL = "camelCase"
    KEBAB = "kebab-case"
    PASCAL = "PascalCase"

In [42]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("JSONRefiner")

In [43]:
class JSONRefinerCore:

    def __init__(self):
        self.errors = []

    def log_error(self, message):
        self.errors.append(message)
        logger.error(message)

    def generate_statistics(self, data):

        total_keys = 0
        nested_keys = 0

        def count_keys(obj, level=0):
            nonlocal total_keys, nested_keys

            if isinstance(obj, dict):
                for k, v in obj.items():
                    total_keys += 1
                    if level > 0:
                        nested_keys += 1
                    count_keys(v, level + 1)

            elif isinstance(obj, list):
                for item in obj:
                    count_keys(item, level)

        count_keys(data)

        top_level_keys = len(data)
        has_nested = nested_keys > 0

        return {
            "Total Keys": total_keys,
            "Top Level Keys": top_level_keys,
            "Nested Keys": nested_keys,
            "Has Nested Objects": has_nested
        }

In [44]:
def infer_type(value):

    value = value.strip().lower()

    # Boolean handling
    if value in ["true","yes","on"]:
        return True

    if value in ["false","no","off"]:
        return False

    # Null handling
    if value in ["null","none","nil","n/a"]:
        return None

    # Integer
    if re.match(r'^-?\d+$', value):
        return int(value)

    # Float
    if re.match(r'^-?\d+\.\d+$', value):
        return float(value)

    # JSON arrays or objects
    try:
        return json.loads(value)
    except:
        pass

    return value

In [45]:
def parse_key_value(text):

    data = {}

    lines = text.split("\n")

    for line in lines:

        if ":" in line:
            key,value = line.split(":",1)

            key = key.strip()
            value = value.strip()

            data[key] = infer_type(value)

    return data

In [46]:
def to_snake_case(text):
    return re.sub(r'\W+','_',text).lower()

def to_camel_case(text):
    parts = re.split(r'\W+',text)
    return parts[0].lower() + ''.join(word.capitalize() for word in parts[1:])

def to_kebab_case(text):
    return re.sub(r'\W+','-',text).lower()

def to_pascal_case(text):
    return ''.join(word.capitalize() for word in re.split(r'\W+',text))

In [47]:
def normalize_key(key,style):

    if style == CaseStyle.SNAKE:
        return to_snake_case(key)

    if style == CaseStyle.CAMEL:
        return to_camel_case(key)

    if style == CaseStyle.KEBAB:
        return to_kebab_case(key)

    if style == CaseStyle.PASCAL:
        return to_pascal_case(key)

    return key

In [48]:
from jsonschema import validate, ValidationError

In [49]:
def validate_json_schema(data, schema):

    try:
        validate(instance=data, schema=schema)
        return "Schema Validation Success"

    except ValidationError as e:
        return f"Schema Validation Error: {e.message}"

In [50]:
sample_schema = {
    "type": "object",
    "properties": {
        "name": {"type":"string"},
        "age": {"type":"number"}
    },
    "required":["name","age"]
}

sample_data = {
    "name":"John",
    "age":28
}

validate_json_schema(sample_data,sample_schema)

'Schema Validation Success'

In [51]:
def check_required_fields(data, required_fields):

    missing = []

    for field in required_fields:
        if field not in data:
            missing.append(field)

    if missing:
        return f"Missing Fields: {missing}"

    return "All required fields present"

In [52]:
data = {"name":"John"}

check_required_fields(data,["name","age"])

"Missing Fields: ['age']"

In [53]:
def flatten_json(data,parent_key='',sep='.'):

    items={}

    for k,v in data.items():

        new_key = parent_key + sep + k if parent_key else k

        if isinstance(v,dict):
            items.update(flatten_json(v,new_key,sep=sep))

        else:
            items[new_key] = v

    return items

In [54]:
def unflatten_json(data):

    result={}

    for key,value in data.items():

        parts = key.split('.')

        d = result

        for part in parts[:-1]:
            if part not in d:
                d[part]={}

            d=d[part]

        d[parts[-1]]=value

    return result

In [55]:
def merge_json_objects(a,b):

    result = a.copy()

    for key,value in b.items():

        if key in result and isinstance(result[key],dict) and isinstance(value,dict):
            result[key] = merge_json_objects(result[key],value)

        else:
            result[key]=value

    return result

In [57]:
def remove_null_values(data):

    return {k:v for k,v in data.items() if v is not None}

In [58]:
import gradio as gr

In [59]:
def refine_json(text, case_style, remove_null, flatten):

    core = JSONRefinerCore()

    data = parse_key_value(text)

    normalized = {}

    for k, v in data.items():
        new_key = normalize_key(k, CaseStyle(case_style))
        normalized[new_key] = v

    if remove_null:
        normalized = remove_null_values(normalized)

    if flatten:
        normalized = flatten_json(normalized)

    stats = core.generate_statistics(normalized)

    stats_output = "\n".join([f"{k}: {v}" for k,v in stats.items()])

    return json.dumps(normalized, indent=2), stats_output

In [60]:
interface = gr.Interface(

    fn=refine_json,

    inputs=[

        gr.Textbox(lines=10,label="Input Key Value Text"),

        gr.Radio(
            choices=[
                "snake_case",
                "camelCase",
                "kebab-case",
                "PascalCase"
            ],
            value="snake_case",
            label="Case Style"
        ),

        gr.Checkbox(label="Remove Null Values"),
        gr.Checkbox(label="Flatten JSON")

    ],

    outputs=[

        gr.Code(label="Refined JSON Output"),
        gr.Textbox(label="Statistics")

    ],

    title="JSON Refiner Advanced Edition",

    description="Convert raw key value text into clean structured JSON"

)

In [61]:
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b72d2ceab985397443.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
